In [ ]:
!pip install transformers accelerate bitsandbytes wandb lm-eval autoawq -q

In [ ]:
import time
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from transformers.utils import quantization_config
from transformers import BitsAndBytesConfig
import gc
from awq import AutoAWQForCausalLM
from transformers import AutoTokenizer

In [ ]:
import wandb
wandb.login()

## Utilisation initiale
Dans ce cas de figure, nous utilisons la config initiale pour lancer Mistral7B c'est à dire avec une quantification en 16bits

In [ ]:
def benchmark(model, tokenizer, prompts, label):
    wandb.init(project="mistral-optimisation", name=label, reinit=True)

    for prompt in prompts:
        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
        start = time.time()
        with torch.no_grad():
            outputs = model.generate(**inputs, max_new_tokens=200)
        elapsed = time.time() - start
        n_tokens = outputs.shape[1] - inputs.input_ids.shape[1]
        tok_s = n_tokens / elapsed
        vram = torch.cuda.memory_allocated() / 1024**2

        print(f"{label} | {n_tokens} tokens | {tok_s:.1f} tok/s | {vram:.0f} MB VRAM")
        wandb.log({"tokens_per_second": tok_s, "vram_mb": vram, "label": label})

    wandb.finish()

In [ ]:
model_id = "mistralai/Mistral-7B-v0.3"
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id, dtype=torch.float16, device_map="cuda")
prompts = [
    "Explique l'IA en 3 phrases.",
    "Quelle est la capitale de la France ?"
]

benchmark(model, tokenizer, prompts, "baseline-fp16")

## Quantification en 8bits


Supprimer le modèle 16float du cache

In [ ]:
del model
torch.cuda.empty_cache()
gc.collect()

In [ ]:
bnb_8bits = BitsAndBytesConfig(load_in_8bit=True)

model_8bit = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_8bits,
    device_map="cuda"
)

In [ ]:
for name, param in model_8bit.named_parameters():
    print(f"{name}: {param.dtype}")
    break  # enlève ce break pour tout voir

In [ ]:
prompts = [
    "Explique l'IA en 3 phrases.",
    "Quelle est la capitale de la France ?"
]

benchmark(model_8bit, tokenizer, prompts, "int8")

## Quantification en 4bits

Nettoyage du modèle pour respecter le stockage offert par colab.

In [ ]:
del model_8bit
torch.cuda.empty_cache()
gc.collect()

In [ ]:
bnb_4bits = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True
)

In [ ]:
model_4bit = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_4bits,
    device_map="cuda"
)
benchmark(model_4bit, tokenizer, prompts, "int4")

In [ ]:
del model_4bit
torch.cuda.empty_cache()
gc.collect()

In [ ]:
model_awq_id = "TheBloke/Mistral-7B-v0.1-AWQ"
tokenizer = AutoTokenizer.from_pretrained(model_id)
model_awq = AutoAWQForCausalLM.from_quantized(model_awq_id, device_map="auto")

def benchmark_awq(model, tokenizer, prompts, label):
    wandb.init(project="mistral-optimisation", name=label, reinit=True, id=wandb.util.generate_id())
    device = next(model.parameters()).device
    for prompt in prompts:
        inputs = tokenizer(prompt, return_tensors="pt").to(device)
        start = time.time()
        with torch.no_grad():
            outputs = model.generate(**inputs, max_new_tokens=200)
        elapsed = time.time() - start
        n_tokens = outputs.shape[1] - inputs.input_ids.shape[1]
        tok_s = n_tokens / elapsed
        vram = torch.cuda.memory_allocated() / 1024**2
        print(f"{label} | {n_tokens} tokens | {tok_s:.1f} tok/s | {vram:.0f} MB VRAM")
        wandb.log({"tokens_per_second": tok_s, "vram_mb": vram})
    wandb.finish()

In [ ]:
prompts = ["Explique l'IA en 3 phrases.", "Quelle est la capitale de la France ?"]
benchmark_awq(model_awq, tokenizer, prompts, "awq-4bit")

In [ ]:
from datasets import load_dataset

dataset = load_dataset("ccdv/arxiv-summarization", split="train[:5000]")

def format_prompt(example):
    return {
        "text": f"""### Article:
{example['article'][:2000]}

### Résumé:
{example['abstract']}"""
    }

dataset = dataset.map(format_prompt)

GPTQ
SqueezeLLM
EXL2